# EDA inicial — Dataset Olist

Objetivo: entender a base antes de desenhar bronze/silver/gold (ver `architecture.md`).

Pontos verificados em cada tabela:
- quantidade de linhas e colunas
- tipagem (dtype lido pelo pandas vs tipo real esperado)
- nulos (quantidade e %)
- duplicatas (linha inteira e chave de negócio)
- outliers (variáveis numéricas, via IQR)
- período de tempo (colunas de data)

E depois: relação entre as tabelas (integridade referencial / chaves órfãs).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)

RAW = Path('../data/raw')

files = {
    'customers': 'olist_customers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'order_payments': 'olist_order_payments_dataset.csv',
    'order_reviews': 'olist_order_reviews_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'category_translation': 'product_category_name_translation.csv',
}

date_cols = {
    'orders': ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
                'order_delivered_customer_date', 'order_estimated_delivery_date'],
    'order_items': ['shipping_limit_date'],
    'order_reviews': ['review_creation_date', 'review_answer_timestamp'],
}

dfs = {}
for name, fname in files.items():
    df = pd.read_csv(RAW / fname)
    for col in date_cols.get(name, []):
        df[col] = pd.to_datetime(df[col], errors='coerce')
    dfs[name] = df

print('Tabelas carregadas:', list(dfs.keys()))

Tabelas carregadas: ['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'products', 'sellers', 'category_translation']


## 1. Visão geral — linhas, colunas, memória

In [2]:
overview = pd.DataFrame({
    'linhas': {k: len(v) for k, v in dfs.items()},
    'colunas': {k: v.shape[1] for k, v in dfs.items()},
    'memoria_mb': {k: round(v.memory_usage(deep=True).sum() / 1e6, 2) for k, v in dfs.items()},
})
overview

,linhas,colunas,memoria_mb
customers,99441,5,27.88
geolocation,1000163,5,135.67
order_items,112650,7,30.98
order_payments,103886,5,17.02
order_reviews,99224,7,29.12
orders,99441,8,25.85
products,32951,9,6.60
sellers,3095,4,0.62
category_translation,71,2,0.01


## 2. Tipagem por tabela

Olhar dtype lido pelo pandas (tudo que não é data já convertida vem como `object` ou `int64`/`float64`).
Atenção especial a: IDs (devem ser string/object, não numérico), datas (convertidas acima), e colunas que deveriam ser categóricas.

In [3]:
for name, df in dfs.items():
    print(f'--- {name} ---')
    print(df.dtypes)
    print()

--- customers ---
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

--- geolocation ---
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

--- order_items ---
order_id                       object
order_item_id                   int64
product_id                     object
seller_id                      object
shipping_limit_date    datetime64[ns]
price                         float64
freight_value                 float64
dtype: object

--- order_payments ---
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

--- order_reviews ---
review_id                          object

## 3. Nulos por tabela

In [4]:
for name, df in dfs.items():
    n = df.isna().sum()
    n = n[n > 0]
    if n.empty:
        print(f'--- {name}: sem nulos ---\n')
        continue
    pct = (n / len(df) * 100).round(2)
    report = pd.DataFrame({'nulos': n, 'pct': pct}).sort_values('nulos', ascending=False)
    print(f'--- {name} ---')
    print(report)
    print()

--- customers: sem nulos ---



--- geolocation: sem nulos ---

--- order_items: sem nulos ---



--- order_payments: sem nulos ---

--- order_reviews ---
                        nulos    pct
review_comment_title    87656  88.34
review_comment_message  58247  58.70

--- orders ---
                               nulos   pct
order_delivered_customer_date   2965  2.98
order_delivered_carrier_date    1783  1.79
order_approved_at                160  0.16

--- products ---
                            nulos   pct
product_category_name         610  1.85
product_name_lenght           610  1.85
product_description_lenght    610  1.85
product_photos_qty            610  1.85
product_weight_g                2  0.01
product_length_cm               2  0.01
product_height_cm               2  0.01
product_width_cm                2  0.01

--- sellers: sem nulos ---

--- category_translation: sem nulos ---



## 4. Duplicatas

Verifica duplicata de linha inteira e duplicata na chave primária esperada de cada tabela.

In [5]:
pk_expected = {
    'customers': 'customer_id',
    'geolocation': None,  # sem PK natural, zip+lat+lng se repete por design
    'order_items': ['order_id', 'order_item_id'],
    'order_payments': ['order_id', 'payment_sequential'],
    'order_reviews': 'review_id',
    'orders': 'order_id',
    'products': 'product_id',
    'sellers': 'seller_id',
    'category_translation': 'product_category_name',
}

for name, df in dfs.items():
    full_dup = df.duplicated().sum()
    pk = pk_expected[name]
    if pk is None:
        pk_dup = 'n/a (sem PK natural)'
    else:
        pk_dup = df.duplicated(subset=pk).sum()
    print(f'{name:22s} | linha duplicada: {full_dup:7d} | PK duplicada ({pk}): {pk_dup}')

customers              | linha duplicada:       0 | PK duplicada (customer_id): 0


geolocation            | linha duplicada:  261831 | PK duplicada (None): n/a (sem PK natural)


order_items            | linha duplicada:       0 | PK duplicada (['order_id', 'order_item_id']): 0
order_payments         | linha duplicada:       0 | PK duplicada (['order_id', 'payment_sequential']): 0


order_reviews          | linha duplicada:       0 | PK duplicada (review_id): 814
orders                 | linha duplicada:       0 | PK duplicada (order_id): 0
products               | linha duplicada:       0 | PK duplicada (product_id): 0
sellers                | linha duplicada:       0 | PK duplicada (seller_id): 0
category_translation   | linha duplicada:       0 | PK duplicada (product_category_name): 0


## 5. Período de tempo (colunas de data)

In [6]:
rows = []
for name, cols in date_cols.items():
    for col in cols:
        s = dfs[name][col]
        rows.append({
            'tabela': name,
            'coluna': col,
            'min': s.min(),
            'max': s.max(),
            'nulos': s.isna().sum(),
        })
pd.DataFrame(rows)

,tabela,coluna,min,max,nulos
0,orders,order_purchase_timestamp,2016-09-04 21:15:19,2018-10-17 17:30:18,0
1,orders,order_approved_at,2016-09-15 12:16:38,2018-09-03 17:40:06,160
2,orders,order_delivered_carrier_date,2016-10-08 10:34:01,2018-09-11 19:48:28,1783
3,orders,order_delivered_customer_date,2016-10-11 13:46:32,2018-10-17 13:22:46,2965
4,orders,order_estimated_delivery_date,2016-09-30 00:00:00,2018-11-12 00:00:00,0
5,order_items,shipping_limit_date,2016-09-19 00:15:34,2020-04-09 22:35:08,0
6,order_reviews,review_creation_date,2016-10-02 00:00:00,2018-08-31 00:00:00,0
7,order_reviews,review_answer_timestamp,2016-10-07 18:32:28,2018-10-29 12:27:35,0


## 6. Status dos pedidos (orders)

In [7]:
dfs['orders']['order_status'].value_counts(dropna=False)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

## 7. Outliers (IQR) nas colunas numéricas de negócio

Usa a regra de Tukey: outlier = fora de `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`. Só para dar uma primeira noção de cauda longa, não para descartar linhas.

In [8]:
def iqr_outliers(s):
    s = s.dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    out = s[(s < low) | (s > high)]
    return len(out), round(len(out) / len(s) * 100, 2), low, high, s.max()

numeric_targets = {
    'order_items': ['price', 'freight_value'],
    'order_payments': ['payment_value', 'payment_installments'],
    'products': ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_photos_qty'],
}

rows = []
for name, cols in numeric_targets.items():
    for col in cols:
        n_out, pct_out, low, high, mx = iqr_outliers(dfs[name][col])
        rows.append({'tabela': name, 'coluna': col, 'n_outliers': n_out, 'pct': pct_out,
                     'limite_inf': round(low, 2), 'limite_sup': round(high, 2), 'max_real': mx})
pd.DataFrame(rows)

,tabela,coluna,n_outliers,pct,limite_inf,limite_sup,max_real
0,order_items,price,8427,7.48,-102.60,277.40,6735.00
1,order_items,freight_value,12134,10.77,0.98,33.25,409.68
2,order_payments,payment_value,7981,7.68,-115.78,344.41,13664.08
3,order_payments,payment_installments,6313,6.08,-3.50,8.50,24.00
4,products,product_weight_g,4551,13.81,-2100.00,4300.00,40425.00
5,products,product_length_cm,1380,4.19,-12.00,68.00,105.00
6,products,product_height_cm,1892,5.74,-11.50,40.50,105.00
7,products,product_width_cm,912,2.77,-7.50,52.50,118.00
8,products,product_photos_qty,849,2.63,-2.00,6.00,20.00


In [9]:
dfs['order_items'][['price', 'freight_value']].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [10]:
dfs['order_payments'][['payment_value', 'payment_installments']].describe()

,payment_value,payment_installments
count,103886.000000,103886.000000
mean,154.100380,2.853349
std,217.494064,2.687051
min,0.000000,0.000000
25%,56.790000,1.000000
50%,100.000000,1.000000
75%,171.837500,4.000000
max,13664.080000,24.000000


In [11]:
dfs['products'][['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']].describe()

,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32949.000000,32949.000000,32949.000000,32949.000000
mean,2276.472488,30.815078,16.937661,23.196728
std,4282.038731,16.914458,13.637554,12.079047
min,0.000000,7.000000,2.000000,6.000000
25%,300.000000,18.000000,8.000000,15.000000
50%,700.000000,25.000000,13.000000,20.000000
75%,1900.000000,38.000000,21.000000,30.000000
max,40425.000000,105.000000,105.000000,118.000000


## 8. Relação entre tabelas — integridade referencial

Modelo esperado:
```
customers 1---* orders 1---* order_items *---1 products
                  |                |
                  |                *---1 sellers
                  1---* order_payments
                  1---* order_reviews
customers.customer_zip_code_prefix ~ geolocation.geolocation_zip_code_prefix
sellers.seller_zip_code_prefix     ~ geolocation.geolocation_zip_code_prefix
products.product_category_name    ~ category_translation.product_category_name
```
Checar chaves órfãs (FK que não existe na tabela pai) nos dois sentidos relevantes.

In [12]:
def orphan_check(child_df, child_key, parent_df, parent_key, label):
    child_keys = set(child_df[child_key].dropna())
    parent_keys = set(parent_df[parent_key].dropna())
    orphans = child_keys - parent_keys
    print(f'{label}: {len(orphans)} valores órfãos de {len(child_keys)} distintos '
          f'({len(orphans)/max(len(child_keys),1)*100:.2f}%)')
    return orphans

_ = orphan_check(dfs['orders'], 'customer_id', dfs['customers'], 'customer_id', 'orders.customer_id -> customers')
_ = orphan_check(dfs['order_items'], 'order_id', dfs['orders'], 'order_id', 'order_items.order_id -> orders')
_ = orphan_check(dfs['order_items'], 'product_id', dfs['products'], 'product_id', 'order_items.product_id -> products')
_ = orphan_check(dfs['order_items'], 'seller_id', dfs['sellers'], 'seller_id', 'order_items.seller_id -> sellers')
_ = orphan_check(dfs['order_payments'], 'order_id', dfs['orders'], 'order_id', 'order_payments.order_id -> orders')
_ = orphan_check(dfs['order_reviews'], 'order_id', dfs['orders'], 'order_id', 'order_reviews.order_id -> orders')
_ = orphan_check(dfs['products'], 'product_category_name', dfs['category_translation'], 'product_category_name', 'products.product_category_name -> category_translation')
_ = orphan_check(dfs['customers'], 'customer_zip_code_prefix', dfs['geolocation'], 'geolocation_zip_code_prefix', 'customers.zip -> geolocation')
_ = orphan_check(dfs['sellers'], 'seller_zip_code_prefix', dfs['geolocation'], 'geolocation_zip_code_prefix', 'sellers.zip -> geolocation')

orders.customer_id -> customers: 0 valores órfãos de 99441 distintos (0.00%)
order_items.order_id -> orders: 0 valores órfãos de 98666 distintos (0.00%)
order_items.product_id -> products: 0 valores órfãos de 32951 distintos (0.00%)
order_items.seller_id -> sellers: 0 valores órfãos de 3095 distintos (0.00%)
order_payments.order_id -> orders: 0 valores órfãos de 99440 distintos (0.00%)


order_reviews.order_id -> orders: 0 valores órfãos de 98673 distintos (0.00%)
products.product_category_name -> category_translation: 2 valores órfãos de 73 distintos (2.74%)
customers.zip -> geolocation: 157 valores órfãos de 14994 distintos (1.05%)


sellers.zip -> geolocation: 7 valores órfãos de 2246 distintos (0.31%)


### Cardinalidade orders <-> customers

No Olist, `customer_id` é por pedido e `customer_unique_id` identifica a pessoa real. Checar isso.

In [13]:
print('customer_id distintos:', dfs['customers']['customer_id'].nunique())
print('customer_unique_id distintos:', dfs['customers']['customer_unique_id'].nunique())
print('linhas em customers:', len(dfs['customers']))
print('-> cada customer_id é único (1 por pedido); customer_unique_id repete para clientes recorrentes')
print()
print('orders por customer_id (deveria ser sempre 1):')
print(dfs['orders'].groupby('customer_id').size().value_counts())

customer_id distintos: 99441
customer_unique_id distintos: 96096
linhas em customers: 99441
-> cada customer_id é único (1 por pedido); customer_unique_id repete para clientes recorrentes

orders por customer_id (deveria ser sempre 1):
1    99441
Name: count, dtype: int64


### order_items 1 pedido pode ter múltiplos itens — checar grão

In [14]:
items_per_order = dfs['order_items'].groupby('order_id').size()
print(items_per_order.describe())
print()
print('pedidos com mais de 1 item:', (items_per_order > 1).sum(), 'de', len(items_per_order))

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
dtype: float64

pedidos com mais de 1 item: 9803 de 98666


### Pedidos sem itens (orders sem correspondência em order_items)

In [15]:
orders_sem_itens = set(dfs['orders']['order_id']) - set(dfs['order_items']['order_id'])
print('pedidos sem itens:', len(orders_sem_itens))
dfs['orders'][dfs['orders']['order_id'].isin(orders_sem_itens)]['order_status'].value_counts()

pedidos sem itens: 775


order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

## 9. Datas inconsistentes dentro de `orders`

Checar casos onde a sequência lógica de datas é violada (ex.: entregue antes de aprovado, aprovado antes da compra).

In [16]:
o = dfs['orders']
checks = {
    'approved < purchase': (o['order_approved_at'] < o['order_purchase_timestamp']).sum(),
    'delivered_carrier < approved': (o['order_delivered_carrier_date'] < o['order_approved_at']).sum(),
    'delivered_customer < delivered_carrier': (o['order_delivered_customer_date'] < o['order_delivered_carrier_date']).sum(),
    'delivered_customer < purchase': (o['order_delivered_customer_date'] < o['order_purchase_timestamp']).sum(),
}
pd.Series(checks)

approved < purchase                          0
delivered_carrier < approved              1359
delivered_customer < delivered_carrier      23
delivered_customer < purchase                0
dtype: int64

## 10. Resumo (preencher após rodar)

- Linhas por tabela, nulos relevantes, duplicatas, outliers e integridade referencial: ver saídas acima.
- Usar esses achados para fechar as decisões abertas A1–A5 em `architecture.md`.